<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/01-python-foundations/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1: Exercises

Twelve problems, grouped by difficulty: four Easy, five Medium and three Hard. Each one is a function with an empty body. Fill it in, then run the check cell underneath to test it.

**Run the setup cell first.** It defines the check functions. If a check fails, it tells you which input broke your code, what it expected and what you returned, and sometimes gives a hint.

A few rules to get the most out of this:

- Pure Python only. No NumPy, and no imports unless an exercise says otherwise.
- Don't change the function names or arguments, or the checks won't find them.
- Solutions are in `03_solutions.ipynb`. Look at them after you've solved a problem to compare approaches, or if you've been properly stuck for a while.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below. You don't need to read it,
# but feel free to look at the test cases if you're stuck.

import copy
import math


class Raises:
    """Marks a test case where the function is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=60):
    """repr() that stays readable for long lists."""
    text = repr(x)
    if len(text) <= limit:
        return text
    if isinstance(x, tuple):
        return "(" + ", ".join(_short(item, limit) for item in x) + ")"
    if isinstance(x, list):
        head = ", ".join(repr(item) for item in x[:3])
        return f"[{head}, ... {len(x)} items]"
    return text[:limit - 3] + "..."


def _call_str(func, args):
    return f"{func.__name__}({', '.join(_short(a) for a in args)})"


def _run(func, cases):
    for args, expected, hint in cases:
        call = _call_str(func, args)
        before = copy.deepcopy(args)
        try:
            result = func(*args)
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {call}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {expected!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {call}")
            print(f"      expected {expected!r}, but it returned {result!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if args != before:
            print(f"FAIL  {call}")
            print("      your function modified its input. It should leave it unchanged.")
            return
        if not _same(result, expected):
            print(f"FAIL  {call}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {func.__name__}")


def check_accuracy(f):
    _run(f, [
        (([1, 0, 1, 1], [1, 0, 0, 1]), 0.75, None),
        (([1, 1, 1], [1, 1, 1]), 1.0, None),
        (([0, 0], [1, 1]), 0.0, None),
        ((["cat", "dog", "cat"], ["cat", "cat", "cat"]), 2 / 3,
         "labels don't have to be numbers"),
    ])


def check_clip(f):
    _run(f, [
        ((5, 0, 10), 5, None),
        ((-3, 0, 10), 0, None),
        ((15, 0, 10), 10, None),
        ((0.5, 0.0, 1.0), 0.5, None),
        ((1e-12, 1e-7, 1 - 1e-7), 1e-7, None),
    ])


def check_is_valid_probability(f):
    _run(f, [
        ((0.5,), True, None),
        ((0,), True, "0 is a valid probability"),
        ((1,), True, "1 is a valid probability"),
        ((1.0001,), False, None),
        ((-0.1,), False, None),
        ((float("nan"),), False, "NaN is not a valid probability. Check how NaN behaves in comparisons."),
    ])


def check_count_labels(f):
    _run(f, [
        (([1, 0, 1, 1],), {1: 3, 0: 1}, None),
        (([],), {}, "an empty list should give an empty dict"),
        ((["cat", "dog", "cat"],), {"cat": 2, "dog": 1}, None),
    ])


def check_normalize(f):
    _run(f, [
        (([0, 5, 10],), [0.0, 0.5, 1.0], None),
        (([2, 4],), [0.0, 1.0], None),
        (([-1, 0, 1],), [0.0, 0.5, 1.0], None),
        (([3, 3, 3],), [0.0, 0.0, 0.0], "what happens to max - min when all values are equal?"),
        (([10],), [0.0], None),
    ])


def check_mean_and_std(f):
    _run(f, [
        (([1, 2, 3, 4, 5],), (3.0, math.sqrt(2)), "use the population std: divide by n, not n - 1"),
        (([5, 5, 5],), (5.0, 0.0), None),
        (([2, 4, 4, 4, 5, 5, 7, 9],), (5.0, 2.0), None),
        (([-1, 1],), (0.0, 1.0), None),
    ])


def check_one_hot(f):
    _run(f, [
        (("dog", ["cat", "dog", "bird"]), [0, 1, 0], None),
        (("bird", ["cat", "dog", "bird"]), [0, 0, 1], None),
        ((2, [0, 1, 2, 3]), [0, 0, 1, 0], None),
        (("fish", ["cat", "dog", "bird"]), Raises(ValueError), "raise a ValueError for a label that isn't in classes"),
    ])


def check_train_test_split(f):
    _run(f, [
        ((list(range(10)), 0.2), (list(range(8)), [8, 9]), None),
        ((["a", "b", "c", "d"], 0.5), (["a", "b"], ["c", "d"]), None),
        ((list(range(10)), 0.0), (list(range(10)), []), "with test_ratio=0 the test set is empty. Careful: data[-0:] is the whole list"),
        ((list(range(5)), 1.0), ([], list(range(5))), None),
        ((list(range(100)), 0.29), (list(range(71)), list(range(71, 100))),
         "100 * 0.29 is 28.999999999999996 in floating point. Use round(), not int()"),
    ])


def check_confusion_counts(f):
    _run(f, [
        (([1, 0, 1, 1, 0, 0], [1, 1, 0, 1, 0, 0]), {"tp": 2, "fp": 1, "tn": 2, "fn": 1}, None),
        (([1, 1], [1, 1]), {"tp": 2, "fp": 0, "tn": 0, "fn": 0}, "include all four keys, even when a count is 0"),
        (([0, 0, 0], [1, 1, 1]), {"tp": 0, "fp": 3, "tn": 0, "fn": 0}, None),
    ])


def check_precision_recall_f1(f):
    _run(f, [
        (([1, 0, 1, 1, 0, 0], [1, 1, 0, 1, 0, 0]), (2 / 3, 2 / 3, 2 / 3), None),
        (([1, 1, 1, 0], [1, 0, 0, 0]), (1.0, 1 / 3, 0.5), None),
        (([0, 0, 0], [0, 0, 0]), (0.0, 0.0, 0.0), "no positive predictions and no positive labels: return 0.0 instead of dividing by zero"),
        (([1, 0], [0, 1]), (0.0, 0.0, 0.0), "precision and recall are both 0 here, so the F1 formula divides by zero"),
    ])


def check_moving_average(f):
    _run(f, [
        (([1, 2, 3, 4, 5], 2), [1.5, 2.5, 3.5, 4.5], None),
        (([1, 2, 3, 4, 5], 5), [3.0], None),
        (([4, 8, 6], 1), [4.0, 8.0, 6.0], None),
        (([1, 2, 3], 0), Raises(ValueError), "a window of 0 makes no sense"),
        (([1, 2, 3], 4), Raises(ValueError), "the window can't be bigger than the list"),
    ])


def check_batch(f):
    _run(f, [
        ((list(range(7)), 3), [[0, 1, 2], [3, 4, 5], [6]], None),
        ((list(range(7)), 3, True), [[0, 1, 2], [3, 4, 5]], None),
        ((list(range(6)), 2), [[0, 1], [2, 3], [4, 5]], None),
        (([], 4), [], None),
        (([1, 2], 5), [[1, 2]], None),
        (([1, 2], 5, True), [], "the only batch is incomplete, so drop_last removes it"),
        ((list(range(3)), 0), Raises(ValueError), "batch_size must be at least 1"),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `accuracy`

Write `accuracy(y_true, y_pred)` that returns the fraction of predictions that match the true labels.

```python
accuracy([1, 0, 1, 1], [1, 0, 0, 1])  # 0.75
```

You can assume both lists have the same length and aren't empty.

In [ ]:
def accuracy(y_true, y_pred):
    # your code here
    pass

In [ ]:
check_accuracy(accuracy)

### Exercise 2: `clip`

Write `clip(value, low, high)` that returns `value` limited to the range `[low, high]`. If it's below `low`, return `low`. If it's above `high`, return `high`. Otherwise return it unchanged.

```python
clip(15, 0, 10)   # 10
clip(-3, 0, 10)   # 0
```

Where this is used: before computing log loss, libraries clip predicted probabilities to something like `[1e-7, 1 - 1e-7]`, because `log(0)` is negative infinity.

In [ ]:
def clip(value, low, high):
    # your code here
    pass

In [ ]:
check_clip(clip)

### Exercise 3: `is_valid_probability`

Write `is_valid_probability(p)` that returns `True` if `p` is between 0 and 1 (inclusive), and `False` otherwise.

```python
is_valid_probability(0.3)   # True
is_valid_probability(1.2)   # False
```

One of the test cases is `float("nan")`. Think about what should happen.

In [ ]:
def is_valid_probability(p):
    # your code here
    pass

In [ ]:
check_is_valid_probability(is_valid_probability)

### Exercise 4: `count_labels`

Write `count_labels(labels)` that returns a dictionary mapping each label to how many times it appears.

```python
count_labels(["cat", "dog", "cat"])   # {"cat": 2, "dog": 1}
```

Don't import anything. Build the dictionary yourself.

In [ ]:
def count_labels(labels):
    # your code here
    pass

In [ ]:
check_count_labels(count_labels)

## Medium

### Exercise 5: `normalize`

Write `normalize(values)` that scales a list of numbers to the range 0 to 1 using min-max scaling:

$$x' = \frac{x - \min}{\max - \min}$$

```python
normalize([0, 5, 10])   # [0.0, 0.5, 1.0]
```

If all the values are the same, return a list of `0.0` with the same length.

In [ ]:
def normalize(values):
    # your code here
    pass

In [ ]:
check_normalize(normalize)

### Exercise 6: `mean_and_std`

Write `mean_and_std(values)` that returns a tuple `(mean, std)`, where `std` is the population standard deviation:

$$\sigma = \sqrt{\frac{1}{n} \sum_i (x_i - \mu)^2}$$

```python
mean_and_std([2, 4, 4, 4, 5, 5, 7, 9])   # (5.0, 2.0)
```

No imports needed. `x ** 0.5` is the square root of `x`.

In [ ]:
def mean_and_std(values):
    # your code here
    pass

In [ ]:
check_mean_and_std(mean_and_std)

### Exercise 7: `one_hot`

Write `one_hot(label, classes)` that returns a list with a 1 at the position of `label` in `classes` and 0 everywhere else.

```python
one_hot("dog", ["cat", "dog", "bird"])   # [0, 1, 0]
```

If `label` isn't in `classes`, raise a `ValueError`.

In [ ]:
def one_hot(label, classes):
    # your code here
    pass

In [ ]:
check_one_hot(one_hot)

### Exercise 8: `train_test_split`

Write `train_test_split(data, test_ratio)` that splits a list into a training part and a test part, and returns them as a tuple `(train, test)`.

- The number of test items is `round(len(data) * test_ratio)`.
- The test set is taken from the **end** of the list. The training set is everything before it.
- Don't modify the original list.

```python
train_test_split([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 0.2)
# ([0, 1, 2, 3, 4, 5, 6, 7], [8, 9])
```

Make sure it works when `test_ratio` is `0.0` and `1.0`.

In [ ]:
def train_test_split(data, test_ratio):
    # your code here
    pass

In [ ]:
check_train_test_split(train_test_split)

### Exercise 9: `confusion_counts`

For binary classification (labels are 0 or 1, and 1 is the positive class), write `confusion_counts(y_true, y_pred)` that returns a dictionary with four counts:

- `"tp"` (true positive): label 1, predicted 1
- `"fp"` (false positive): label 0, predicted 1
- `"tn"` (true negative): label 0, predicted 0
- `"fn"` (false negative): label 1, predicted 0

```python
confusion_counts([1, 0, 1, 1, 0, 0], [1, 1, 0, 1, 0, 0])
# {"tp": 2, "fp": 1, "tn": 2, "fn": 1}
```

All four keys must be present, even if a count is zero.

In [ ]:
def confusion_counts(y_true, y_pred):
    # your code here
    pass

In [ ]:
check_confusion_counts(confusion_counts)

## Hard

### Exercise 10: `precision_recall_f1`

Write `precision_recall_f1(y_true, y_pred)` that returns a tuple `(precision, recall, f1)` for binary labels:

- precision = tp / (tp + fp): of everything predicted positive, how much really was positive
- recall = tp / (tp + fn): of everything that really was positive, how much we found
- f1 = 2 · precision · recall / (precision + recall)

If any denominator is zero, use `0.0` for that value instead of crashing.

You can use your `confusion_counts` function from the previous exercise.

In [ ]:
def precision_recall_f1(y_true, y_pred):
    # your code here
    pass

In [ ]:
check_precision_recall_f1(precision_recall_f1)

### Exercise 11: `moving_average`

Write `moving_average(values, window)` that returns the average of every run of `window` consecutive values.

```python
moving_average([1, 2, 3, 4, 5], 2)   # [1.5, 2.5, 3.5, 4.5]
```

The result has `len(values) - window + 1` items. Raise a `ValueError` if `window` is less than 1 or bigger than `len(values)`.

This is how training curves get smoothed in tools like TensorBoard, since per-batch loss is very noisy.

Bonus (not tested): the simple version recomputes a full sum for every window. Can you do it with a single running total that you update as the window slides?

In [ ]:
def moving_average(values, window):
    # your code here
    pass

In [ ]:
check_moving_average(moving_average)

### Exercise 12: `batch`

Write `batch(items, batch_size, drop_last=False)` that splits a list into consecutive chunks of `batch_size` items.

```python
batch([0, 1, 2, 3, 4, 5, 6], 3)         # [[0, 1, 2], [3, 4, 5], [6]]
batch([0, 1, 2, 3, 4, 5, 6], 3, True)   # [[0, 1, 2], [3, 4, 5]]
```

- The last batch can be smaller than `batch_size`, unless `drop_last` is `True`, in which case an incomplete last batch is removed.
- Raise a `ValueError` if `batch_size` is less than 1.

Mini-batches are how almost every model is trained. Instead of computing gradients on the whole dataset at once, you go through it a batch at a time.

In [ ]:
def batch(items, batch_size, drop_last=False):
    # your code here
    pass

In [ ]:
check_batch(batch)

## Done?

If every check prints PASS, nice work. Compare your answers with `03_solutions.ipynb`, then move on to the mini project in `04_mini_project.ipynb`, where you'll use several of these functions to build a classifier from scratch.